# Upscale — learned model (PyTorch)

The smallest thing that counts as a learned upscaler, trained on what
`data_preprocess.ipynb` wrote. It exists to make the rest of the work measurable: a
number to beat, a runtime to fit inside a frame budget, a checkpoint format to keep, and
an inference path from a PNG to a PNG.

The bar is in `upscale_dummy.ipynb` — the static resampling filters, which need no model
and cost a fraction of a millisecond. Beating them on PSNR is not the test; beating them
on PSNR *inside* a frame budget they leave almost entirely unspent is.

The model is ESPCN-shaped — three convolutions at *low* resolution and one pixel shuffle
at the end. Upscaling last is the whole point: a network that upsamples first does every
convolution over `SCALE**2` times as many pixels for the same result, which on a live
desktop stream is the difference between keeping up and not.

It learns a **residual on top of bicubic**, and the last convolution starts at zero. So
epoch 0 *is* the bicubic baseline, and every PSNR the training loop prints is the honest
gain over it — a plain network would spend its first epochs relearning interpolation and
the curve would say nothing about whether it works.

In [ ]:
"""Configuration."""

from pathlib import Path

DATA_DIR = Path("data/processed")
CHECKPOINT_DIR = Path("checkpoints")
SAMPLE_DIR = Path("data/samples")

CHANNELS = 64                  # feature width of the first convolution
EPOCHS = 12
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0

In [ ]:
"""Imports, device, and the dataset written by the preprocessing notebook."""

import json
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "manifest.json").exists():
    raise RuntimeError(f"no dataset in {DATA_DIR} — run data_preprocess.ipynb first")

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
SCALE = manifest["scale"]

print("device ", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("torch  ", torch.__version__)
print("scale  ", SCALE)
for split, info in manifest["splits"].items():
    print(f"{split:5}  {info['patches']:5} patches from {len(info['frames'])} frames")

## Dataset

The `.npy` files are memory mapped rather than read: the arrays are small today, but the
access pattern is one random patch at a time, which is exactly what a memory map is for,
and it keeps working when the set outgrows RAM.

The only augmentation is the eight-way dihedral group — flips and 90° rotations. They are
free, they are exact (no resampling, so no new degradation is invented), and they are
applied to LR and HR with the same parameters or the pair stops being a pair.

In [ ]:
"""Patch pairs as float tensors in [0, 1], with the flip/rotate augmentation."""

class PatchPairs(Dataset):
    def __init__(self, directory, split, manifest, augment):
        info = manifest["splits"][split]
        self.lr = np.load(directory / info["lr"], mmap_mode="r")
        self.hr = np.load(directory / info["hr"], mmap_mode="r")
        self.augment = augment

    def __len__(self):
        return len(self.lr)

    def __getitem__(self, index):
        lr = np.asarray(self.lr[index])
        hr = np.asarray(self.hr[index])

        if self.augment:
            turns = int(torch.randint(0, 4, ()))
            lr, hr = np.rot90(lr, turns), np.rot90(hr, turns)
            if int(torch.randint(0, 2, ())):
                lr, hr = lr[:, ::-1], hr[:, ::-1]

        to_tensor = lambda a: torch.from_numpy(
            np.ascontiguousarray(a.transpose(2, 0, 1))).float().div_(255.0)
        return to_tensor(lr), to_tensor(hr)


train_set = PatchPairs(DATA_DIR, "train", manifest, augment=True)
val_set = PatchPairs(DATA_DIR, "val", manifest, augment=False)

# num_workers stays 0: the patches are already in memory-mapped arrays, and on Windows a
# worker process would re-import this notebook module to get there.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

lr_batch, hr_batch = next(iter(train_loader))
print("lr batch", tuple(lr_batch.shape), float(lr_batch.min()), float(lr_batch.max()))
print("hr batch", tuple(hr_batch.shape))

## Model and metric

PSNR is reported on RGB in [0, 1] against the clamped output. It is a poor judge of how
an upscale *looks*, but it is the right judge of whether training is doing anything at
all, which is all this notebook claims to answer.

In [ ]:
"""The upscaler, the bicubic baseline it is measured against, and PSNR."""

class ResidualUpscaler(nn.Module):
    def __init__(self, scale, channels=64):
        super().__init__()
        self.scale = scale
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels // 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.residual = nn.Conv2d(channels // 2, 3 * scale * scale, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)

        # Start as the identity on top of bicubic: no gradient signal is wasted on
        # relearning interpolation, and epoch 0 measures exactly the baseline.
        nn.init.zeros_(self.residual.weight)
        nn.init.zeros_(self.residual.bias)

    def forward(self, x):
        base = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)
        return base + self.shuffle(self.residual(self.features(x)))


def bicubic(x, scale):
    return F.interpolate(x, scale_factor=scale, mode="bicubic", align_corners=False)


def psnr(prediction, target):
    mse = F.mse_loss(prediction.clamp(0, 1), target, reduction="none").mean(dim=(1, 2, 3))
    return (10.0 * torch.log10(1.0 / mse.clamp_min(1e-12))).mean().item()


model = ResidualUpscaler(SCALE, CHANNELS).to(DEVICE)
parameters = sum(p.numel() for p in model.parameters())
print(model)
print(f"\n{parameters:,} parameters ({parameters * 4 / 1e6:.2f} MB as float32)")

In [ ]:
"""Average PSNR over a loader, for the model and for the bicubic baseline."""

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    scores, baselines, count = 0.0, 0.0, 0
    for lr, hr in loader:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        n = lr.size(0)
        scores += psnr(model(lr), hr) * n
        baselines += psnr(bicubic(lr, SCALE), hr) * n
        count += n
    return scores / count, baselines / count


start_model, start_baseline = evaluate(model, val_loader)
print(f"before training  model {start_model:.2f} dB   bicubic {start_baseline:.2f} dB")
print("the two match because the residual starts at zero")

## Training

L1 rather than L2: the same networks trained on L1 come out visibly sharper, because a
squared error is minimised by hedging towards the mean of the plausible outputs, which is
a blur. The loss is still reported next to PSNR, which is an L2 measure — they are
allowed to disagree.

In [ ]:
"""Train, tracking validation PSNR each epoch and keeping the best weights."""

optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

history = {"loss": [], "psnr": [], "baseline": []}
best = {"psnr": float("-inf"), "epoch": -1, "state": None}

for epoch in range(EPOCHS):
    model.train()
    started, running, seen = time.time(), 0.0, 0

    for lr, hr in train_loader:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        loss = F.l1_loss(model(lr), hr)

        optimiser.zero_grad(set_to_none=True)
        loss.backward()
        optimiser.step()

        running += loss.item() * lr.size(0)
        seen += lr.size(0)

    schedule.step()
    train_loss = running / seen
    val_psnr, val_baseline = evaluate(model, val_loader)

    history["loss"].append(train_loss)
    history["psnr"].append(val_psnr)
    history["baseline"].append(val_baseline)

    if val_psnr > best["psnr"]:
        best = {"psnr": val_psnr, "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}

    print(f"epoch {epoch + 1:2}/{EPOCHS}  l1 {train_loss:.4f}  "
          f"val {val_psnr:.2f} dB  bicubic {val_baseline:.2f} dB  "
          f"gain {val_psnr - val_baseline:+.2f} dB  {time.time() - started:.1f}s")

model.load_state_dict(best["state"])
print(f"\nbest epoch {best['epoch'] + 1} at {best['psnr']:.2f} dB, restored")

In [ ]:
"""Loss and PSNR against the baseline."""

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = range(1, len(history["loss"]) + 1)

left.plot(epochs, history["loss"])
left.set_title("training L1 loss")
left.set_xlabel("epoch")

right.plot(epochs, history["psnr"], label="model")
right.plot(epochs, history["baseline"], "--", label="bicubic")
right.set_title("validation PSNR (dB)")
right.set_xlabel("epoch")
right.legend()

fig.tight_layout()
plt.show()

## What it looks like

The numbers say the model beats bicubic; this says where. Look at the text-like rows and
the window edges, which is where a screen stream lives — a gain that shows up only on the
noise block is a model that has learned to sharpen randomness.

In [ ]:
"""Nearest / bicubic / model / target, on validation patches."""

@torch.no_grad()
def compare(model, dataset, count=4):
    model.eval()
    picks = np.random.default_rng(SEED).choice(len(dataset), min(count, len(dataset)), replace=False)
    lr = torch.stack([dataset[int(i)][0] for i in picks]).to(DEVICE)
    hr = torch.stack([dataset[int(i)][1] for i in picks]).to(DEVICE)

    columns = {
        "nearest": F.interpolate(lr, scale_factor=SCALE, mode="nearest"),
        "bicubic": bicubic(lr, SCALE),
        "model": model(lr),
        "target": hr,
    }

    fig, axes = plt.subplots(len(picks), 4, figsize=(11, 2.8 * len(picks)))
    axes = np.atleast_2d(axes)
    for row in range(len(picks)):
        for column, (name, images) in enumerate(columns.items()):
            image = images[row].clamp(0, 1).cpu().permute(1, 2, 0).numpy()
            axes[row][column].imshow(image, interpolation="nearest")
            axes[row][column].axis("off")
            if row == 0:
                axes[row][column].set_title(name)
    fig.tight_layout()
    plt.show()


compare(model, val_set)

## Checkpoint

The weights alone are not a model — the scale factor, the feature width and the
degradation they were trained for all have to come back with them, or inference silently
reconstructs the wrong architecture.

In [ ]:
"""Save the weights next to everything needed to rebuild the model."""

checkpoint_path = CHECKPOINT_DIR / "upscale_nn.pt"
torch.save({
    "architecture": "ResidualUpscaler",
    "state_dict": model.state_dict(),
    "scale": SCALE,
    "channels": CHANNELS,
    "epochs": EPOCHS,
    "best_epoch": best["epoch"] + 1,
    "val_psnr": best["psnr"],
    "val_psnr_bicubic": history["baseline"][-1],
    "degradation": manifest["degradation"],
    "trained": time.strftime("%Y-%m-%dT%H:%M:%S"),
}, checkpoint_path)

print(f"{checkpoint_path.resolve()}  ({checkpoint_path.stat().st_size / 1e6:.2f} MB)")

## Inference on a whole frame

Patches were a training convenience; the model is fully convolutional, so it runs on a
frame of any size in one pass. Whole frames only work while they fit in memory — a 4K
input needs tiling with a few pixels of overlap, and that is the next piece of real work.

In [ ]:
"""Load the checkpoint back and upscale one real image with it."""

def load_upscaler(path, device):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    model = ResidualUpscaler(checkpoint["scale"], checkpoint["channels"]).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint


@torch.no_grad()
def upscale_image(model, image, device):
    x = torch.from_numpy(np.asarray(image.convert("RGB")).transpose(2, 0, 1))
    x = x.float().div_(255.0).unsqueeze(0).to(device)
    y = model(x).clamp(0, 1).squeeze(0).mul(255.0).round().byte()
    return Image.fromarray(y.cpu().permute(1, 2, 0).numpy())


loaded, checkpoint = load_upscaler(checkpoint_path, DEVICE)
print({k: v for k, v in checkpoint.items() if k != "state_dict"})

# Downscale a source frame, then upscale it back, so there is a ground truth to compare to.
source = sorted(Path("data/raw").glob("*.png"))[0]
truth = Image.open(source).convert("RGB")
truth = truth.crop((0, 0, truth.width - truth.width % SCALE, truth.height - truth.height % SCALE))
small = truth.resize((truth.width // SCALE, truth.height // SCALE), Image.BICUBIC)

started = time.time()
restored = upscale_image(loaded, small, DEVICE)
elapsed = time.time() - started

restored.save(SAMPLE_DIR / "upscaled.png")
small.resize(truth.size, Image.BICUBIC).save(SAMPLE_DIR / "bicubic.png")

print(f"{source.name}  {small.size} -> {restored.size} in {elapsed * 1000:.0f} ms "
      f"({1 / elapsed:.1f} fps at this size on {DEVICE.type})")
print("written to", SAMPLE_DIR.resolve())

In [ ]:
"""The same crop through both paths, at 1:1 pixels."""

box = (0, 0, min(320, truth.width), min(200, truth.height))
panels = {
    "bicubic": small.resize(truth.size, Image.BICUBIC).crop(box),
    f"model ({checkpoint['val_psnr']:.2f} dB)": restored.crop(box),
    "target": truth.crop(box),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (name, image) in zip(axes, panels.items()):
    ax.imshow(image, interpolation="nearest")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
fig.tight_layout()
plt.show()

as_array = lambda image: np.asarray(image, dtype=np.float32) / 255.0
mse = lambda a, b: float(np.mean((a - b) ** 2))
truth_array = as_array(truth)
print(f"full frame PSNR  bicubic "
      f"{10 * np.log10(1 / mse(as_array(small.resize(truth.size, Image.BICUBIC)), truth_array)):.2f} dB"
      f"   model {10 * np.log10(1 / mse(as_array(restored), truth_array)):.2f} dB")

## Runtime

For a live stream the runtime is not a footnote to the PSNR, it is half the result: an
upscaler that cannot clear the frame budget is not an upscaler, whatever it scores. Two
measurements, and they answer different questions.

**Batch of one** is the honest number for a stream. Frames arrive one at a time and each
one is wanted now, so there is nothing to batch with — the per-frame cost at batch 1 is
what has to fit inside the frame interval (16.7 ms at 60 fps, 33.3 ms at 30 fps),
alongside decoding, drawing and everything else the client is doing.

**The batch sweep** is what a batch buys. Larger batches amortise the per-call overhead
and keep a wide device fed, so the cost *per frame* can fall — while the latency of any
one frame rises with the batch it waits in. But a CPU already spreading a single frame
across every core has nothing left to win, and then a batch is pure added latency. Which
of the two happens is why this is measured rather than assumed.

Timings are a forward pass only — no data loading, no PNG encoding, and on CUDA they are
taken after a synchronise, since the queue would otherwise return before the work is done.

In [ ]:
"""Time one forward pass at a given batch size and input size."""

BENCH_LR_SIZE = (320, 180)                  # input for the batch sweep, kept small
BENCH_BATCHES = [1, 2, 4, 8, 16]
BENCH_RESOLUTIONS = [(480, 270), (640, 360), (960, 540), (1280, 720)]


@torch.no_grad()
def benchmark(model, batch, lr_size, device, seconds=1.0, warmup=2):
    model.eval()
    width, height = lr_size
    x = torch.rand(batch, 3, height, width, device=device)

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    for _ in range(warmup):
        model(x)
    sync()

    # One timed pass sets how many the measurement can afford inside `seconds`.
    started = time.perf_counter()
    model(x)
    sync()
    once = time.perf_counter() - started

    reps = max(3, min(50, int(seconds / max(once, 1e-6))))
    started = time.perf_counter()
    for _ in range(reps):
        model(x)
    sync()
    elapsed = (time.perf_counter() - started) / reps

    return {
        "batch": batch,
        "in": f"{width}x{height}",
        "out": f"{width * SCALE}x{height * SCALE}",
        "reps": reps,
        "ms_batch": elapsed * 1000,
        "ms_frame": elapsed * 1000 / batch,
        "fps": batch / elapsed,
    }


print(f"{DEVICE.type}, {torch.get_num_threads()} threads" if DEVICE.type == "cpu"
      else f"{torch.cuda.get_device_name(0)}")

In [ ]:
"""Batch of one, at the resolutions a client would actually ask for."""

single = [benchmark(loaded, 1, size, DEVICE) for size in BENCH_RESOLUTIONS]

print(f"{'output':>12}  {'ms/frame':>9}  {'fps':>7}   frame budget")
for row in single:
    budget = ("60 fps" if row["ms_frame"] <= 16.7 else
              "30 fps" if row["ms_frame"] <= 33.3 else
              "under 30 fps")
    print(f"{row['out']:>12}  {row['ms_frame']:9.1f}  {row['fps']:7.1f}   {budget}")

In [ ]:
"""What a batch buys: per-frame cost against the latency of the batch itself."""

sweep = [benchmark(loaded, batch, BENCH_LR_SIZE, DEVICE) for batch in BENCH_BATCHES]

print(f"input {sweep[0]['in']} -> {sweep[0]['out']}\n")
print(f"{'batch':>5}  {'ms/batch':>9}  {'ms/frame':>9}  {'fps':>8}  {'vs batch 1':>10}")
for row in sweep:
    print(f"{row['batch']:5}  {row['ms_batch']:9.1f}  {row['ms_frame']:9.2f}  "
          f"{row['fps']:8.1f}  {sweep[0]['ms_frame'] / row['ms_frame']:9.2f}x")

In [ ]:
"""The same sweep as a picture: per-frame cost, batch latency, and throughput."""

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
batches = [row["batch"] for row in sweep]

left.plot(batches, [row["ms_frame"] for row in sweep], "o-", label="per frame")
left.plot(batches, [row["ms_batch"] for row in sweep], "o--", label="per batch (latency)")
left.set_xscale("log", base=2)
left.set_yscale("log")
left.set_xlabel("batch size")
left.set_ylabel("ms")
left.set_title("cost")
left.legend()

right.plot(batches, [row["fps"] for row in sweep], "o-")
right.set_xscale("log", base=2)
right.set_xlabel("batch size")
right.set_ylabel("frames per second")
right.set_title("throughput")

fig.suptitle(f"{sweep[0]['in']} -> {sweep[0]['out']} on {DEVICE.type}")
fig.tight_layout()
plt.show()

## Export to ONNX for the browser

The checkpoint above is a torch artefact and the client is a browser, so the model has to
leave here as ONNX too, pinned to the tile the benchmark page runs, and published beside
the other candidates. A model that is only ever measured on this machine is a model
nobody can choose against.

**This one exports badly, and that is worth seeing rather than hiding.** The residual sits
on top of a *bicubic* upsample, which is `F.interpolate` here and becomes `Resize` with
`mode="cubic"` in the graph — an operator the WebGPU provider is least likely to run
natively, in the middle of a graph that is otherwise plain convolutions. A fallback there
is not a slower kernel; it copies the tensor out of GPU memory to the CPU and back, once
per frame. The cell prints the operator list and flags anything outside the budget, so the
problem is visible in the output rather than in a comment.

The fix is a skip connection that is not a `Resize` at all — a frozen 1×1 convolution
feeding `DepthToSpace` is bit-exact nearest-neighbour upsampling — and it belongs in the
notebook of whichever model is built that way, not here. This one stays on the page as the
measurement of what the expensive skip costs.

The halo is set by the receptive field: 5×5 then 3×3 then 3×3 is 4 LR pixels down the
residual path, and the bicubic base reaches 2, so 4 covers both.

In [ ]:
"""Export the trained model for the browser and publish it to the benchmark page."""

import onnxruntime as ort

import webexport

HALO = 4                                    # 5x5 -> 2, then two 3x3 -> 1 each
size = webexport.TILE_STEP + 2 * HALO

exported, _ = load_upscaler(checkpoint_path, torch.device("cpu"))
info = webexport.export(exported, f"upscale_nn_tile{webexport.TILE_STEP}.onnx", size,
                        label="nn - bicubic residual (has Resize)", halo=HALO)
path = info["path"]

# The graph has to compute what the checkpoint computes, or the page benchmarks a model
# that no measurement in this notebook applies to.
options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
runtime = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])

probe = torch.rand(1, 3, size, size)
expected = exported(probe).detach().numpy()
actual = runtime.run(None, {"input": probe.numpy()})[0]
print(f"torch vs onnxruntime: max abs difference {np.abs(expected - actual).max():.2e}")

sample = np.random.rand(1, 3, size, size).astype(np.float32)
runtime.run(None, {"input": sample})
started = time.perf_counter()
for _ in range(20):
    runtime.run(None, {"input": sample})
cpu_ms = (time.perf_counter() - started) / 20 * 1000

print(f"{size}x{size} -> {size * SCALE}x{size * SCALE}, keeps "
      f"{webexport.TILE_STEP * SCALE}x{webexport.TILE_STEP * SCALE}   "
      f"{info['kb']} KB   {cpu_ms:.2f} ms/tile on the desktop CPU runtime")
print(f"operators: {', '.join(f'{op}x{n}' for op, n in sorted(info['ops'].items()))}")
if info["outside_budget"]:
    print(f"outside the WebGPU op budget: {', '.join(info['outside_budget'])}"
          f"  <- expect a CPU fallback in the browser")

# The page reads this back out of the file: there is no manifest to write it to. The
# parameter count and the operator list it also shows come out of the graph itself.
webexport.annotate(info, cpu_ms=round(cpu_ms, 2))
print(f"written to {path.resolve()}")

To see what that costs, serve the page and run this model against the static filters
published by `upscale_dummy.ipynb`:

```
uv run upscale/benchmark/main.py       # from model/, then open http://127.0.0.1:8000
```